In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

In [3]:
crop = pd.read_csv("crop_recommendation-1.csv")

print(crop.head())

     N    P    K  temperature  humidity    ph  rainfall      label
0  102   41  158        12.29     74.50  7.25    164.27     cotton
1   92   13   94        38.22     35.71  5.52    157.53  muskmelon
2   14  103  166        22.50     39.02  8.45    298.97   mungbean
3  106   52  119        27.43     30.18  5.68    271.28     banana
4   71  135  109        37.58     37.59  3.71    149.75     cotton


In [17]:
print(crop.shape)
print(crop.info())
print(crop.isnull().sum())

(200, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            200 non-null    int64  
 1   P            200 non-null    int64  
 2   K            200 non-null    int64  
 3   temperature  200 non-null    float64
 4   humidity     200 non-null    float64
 5   ph           200 non-null    float64
 6   rainfall     200 non-null    float64
 7   label        200 non-null    object 
dtypes: float64(4), int64(3), object(1)
memory usage: 12.6+ KB
None
N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64


In [18]:
X = crop[['N','P','K','temperature','humidity','ph','rainfall']]
y = crop['label']

In [19]:
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [21]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
print("KNN Accuracy:", knn_acc)

KNN Accuracy: 0.0


In [22]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print("Logistic Regression Accuracy:", lr_acc)

Logistic Regression Accuracy: 0.0


C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [23]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
print("Decision Tree Accuracy:", dt_acc)

Decision Tree Accuracy: 0.0


In [24]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.0


In [25]:
kmeans = KMeans(
    n_clusters=len(np.unique(y)),
    random_state=42,
    n_init=10
)

kmeans.fit(X)
print("K-Means Model Trained Successfully")

C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


K-Means Model Trained Successfully


In [26]:
results = pd.DataFrame({
    "Model": [
        "KNN",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        knn_acc,
        lr_acc,
        dt_acc,
        rf_acc
    ]
})

print(results)

best_model = results.loc[results['Accuracy'].idxmax()]
print("\nBest Model:")
print(best_model)

                 Model  Accuracy
0                  KNN       0.0
1  Logistic Regression       0.0
2        Decision Tree       0.0
3        Random Forest       0.0

Best Model:
Model       KNN
Accuracy    0.0
Name: 0, dtype: object


In [27]:
best = rf    # Change if another model performs better

pickle.dump(best, open("crop_model.pkl", "wb"))
print("Model saved successfully.")

Model saved successfully.


In [28]:
model = pickle.load(open("crop_model.pkl", "rb"))

sample = [[90,42,43,20.8,82,6.5,202]]

prediction = model.predict(sample)

crop_name = encoder.inverse_transform(prediction)

print("Recommended Crop:", crop_name[0])

Recommended Crop: papaya


C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [42]:
import pandas as pd

def load_data(path):
    data = pd.read_csv(path)
    data = data.drop_duplicates()
    data = data.dropna()
    return data

In [43]:
def validate_input(N, P, K, temperature, humidity, ph, rainfall):

    values = [N, P, K, temperature, humidity, ph, rainfall]

    for value in values:
        if value is None:
            return False

    return True

In [44]:
import pickle
import numpy as np

model = pickle.load(open("crop_model.pkl", "rb"))

crop_names = [
    "apple","banana","blackgram","chickpea","coconut",
    "coffee","cotton","grapes","jute","kidneybeans",
    "lentil","maize","mango","mothbeans","mungbean",
    "muskmelon","orange","papaya","pigeonpeas","pomegranate",
    "rice","watermelon"
]

def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):

    data = np.array([[N, P, K, temperature, humidity, ph, rainfall]])

    prediction = model.predict(data)

    return crop_names[int(prediction[0])]

In [46]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

data = pd.read_csv("crop_recommendation-1.csv")

X = data[['N','P','K','temperature','humidity','ph','rainfall']]

y = data['label']

encoder = LabelEncoder()

y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pickle.dump(model, open("crop_model.pkl","wb"))
pickle.dump(encoder, open("label_encoder.pkl","wb"))

print("Model Saved Successfully")

Model Saved Successfully


In [47]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [48]:
crop = pd.read_csv("crop_recommendation-1.csv")

print(crop.head())

     N    P    K  temperature  humidity    ph  rainfall      label
0  102   41  158        12.29     74.50  7.25    164.27     cotton
1   92   13   94        38.22     35.71  5.52    157.53  muskmelon
2   14  103  166        22.50     39.02  8.45    298.97   mungbean
3  106   52  119        27.43     30.18  5.68    271.28     banana
4   71  135  109        37.58     37.59  3.71    149.75     cotton


In [49]:
# Check missing values
print(crop.isnull().sum())

# Remove missing values
crop = crop.dropna()

# Remove duplicate records
crop = crop.drop_duplicates()

N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64


In [50]:
crop = crop.fillna(crop.mean(numeric_only=True))

In [51]:
X = crop[['N','P','K','temperature','humidity','ph','rainfall']]

y = crop['label']

In [52]:
encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(X)

In [54]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [55]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [56]:
pickle.dump(model, open("crop_model.pkl", "wb"))

print("Model Saved Successfully")

Model Saved Successfully


In [57]:
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("Scaler Saved Successfully")

Scaler Saved Successfully


In [58]:
pickle.dump(encoder, open("label_encoder.pkl", "wb"))

print("Label Encoder Saved Successfully")

Label Encoder Saved Successfully


In [59]:
model = pickle.load(open("crop_model.pkl", "rb"))

scaler = pickle.load(open("scaler.pkl", "rb"))

encoder = pickle.load(open("label_encoder.pkl", "rb"))

In [60]:
sample = [[90,42,43,20.8,82,6.5,202]]

sample = scaler.transform(sample)

prediction = model.predict(sample)

crop_name = encoder.inverse_transform(prediction)

print("Recommended Crop:", crop_name[0])

Recommended Crop: papaya


C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [61]:
processed = pd.DataFrame(
    X,
    columns=['N','P','K','temperature','humidity','ph','rainfall']
)

processed["label"] = y

processed.to_csv("processed_crop_dataset.csv", index=False)

print("Processed dataset saved.")

Processed dataset saved.


In [62]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.0

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       3.0
           2       0.00      0.00      0.00       1.0
           3       0.00      0.00      0.00       2.0
           4       0.00      0.00      0.00       3.0
           5       0.00      0.00      0.00       3.0
           6       0.00      0.00      0.00       1.0
           7       0.00      0.00      0.00       1.0
           8       0.00      0.00      0.00       2.0
           9       0.00      0.00      0.00       1.0
          10       0.00      0.00      0.00       1.0
          11       0.00      0.00      0.00       1.0
          12       0.00      0.00      0.00       1.0
          13       0.00      0.00      0.00       2.0
          14       0.00      0.00      0.00       2.0
          15       0.00      0.00      0.00       3.0
          16       0.00      0.00      0.0

C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [63]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("Cross Validation Scores:", scores)
print("Average Accuracy:", scores.mean())

C:\Users\sreen\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


Cross Validation Scores: [0.1   0.05  0.025 0.025 0.025]
Average Accuracy: 0.045
